In [1]:
import sys
!{sys.executable} -m pip install deap numpy


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: /Users/vikasshukla/.pyenv/versions/3.11.6/bin/python -m pip install --upgrade pip


In [15]:
import random
import numpy as np
from functools import partial

from deap import algorithms, base, creator, tools, gp

In [16]:
target_map = """
S##.............................
..#.............................
..######.........##########.....
.......#..................#.....
.......#.................#......
.......#####.......######.......
...........#......#.............
...........#......#.............
...........#......#.............
.....#######......#.............
.....#............#.............
.....#............###...........
.....#..............#...........
......#.............#...........
......#.............#...........
.......#.....#......#...........
.......#............#...........
.........#..........#...........
.........#..#.......#...........
.........#..#.......#...........
.........#..#........#.........
.........#..#........#.........
..##..######..#......#.........
.#.............#......#........
.#.............#......#........
.#.....######.........#........
.#....#...............#........
......#.............#..........
.#####.........#.....#.........
...............######..........
"""

In [17]:
class RobotController:

    def __init__(self, max_moves):
        self.max_moves = max_moves
        self.moves = 0
        self.consumed = 0
        self.direction = 1

    def traverse_map(self, matrix):
        rows = [list(row) for row in matrix.strip().split("\n")]
        max_cols = max(len(row) for row in rows)

        self.matrix = [row + ["."] * (max_cols - len(row)) for row in rows]

        self.matrix_row = len(self.matrix)
        self.matrix_col = len(self.matrix[0])

        self.direction_row = [-1, 0, 1, 0]
        self.direction_col = [0, 1, 0, -1]

        self.row_start = None
        self.col_start = None

        for i, row in enumerate(self.matrix):
            for j, value in enumerate(row):
                if value == "S":
                    self.row_start = i
                    self.col_start = j
                    self.matrix[i][j] = "."

        if self.row_start is None or self.col_start is None:
            raise ValueError("Start position S was not found in the map.")

    def _reset(self):
        self.row = self.row_start
        self.col = self.col_start
        self.moves = 0
        self.consumed = 0
        self.direction = 1
        self.matrix_exc = [row[:] for row in self.matrix]

    def turn_left(self):
        if self.moves < self.max_moves:
            self.moves += 1
            self.direction = (self.direction - 1) % 4

    def turn_right(self):
        if self.moves < self.max_moves:
            self.moves += 1
            self.direction = (self.direction + 1) % 4

    def move_forward(self):
        if self.moves < self.max_moves:
            self.moves += 1

            row_new = self.row + self.direction_row[self.direction]
            col_new = self.col + self.direction_col[self.direction]

            if 0 <= row_new < self.matrix_row and 0 <= col_new < self.matrix_col:
                self.row = row_new
                self.col = col_new

                if self.matrix_exc[self.row][self.col] == "#":
                    self.consumed += 1
                    self.matrix_exc[self.row][self.col] = "."

    def sense_target(self):
        row_new = self.row + self.direction_row[self.direction]
        col_new = self.col + self.direction_col[self.direction]

        if 0 <= row_new < self.matrix_row and 0 <= col_new < self.matrix_col:
            return self.matrix_exc[row_new][col_new] == "#"

        return False

    def _conditional(self, condition, out1, out2):
        if condition():
            out1()
        else:
            out2()

    def if_target_ahead(self, out1, out2):
        return partial(self._conditional, self.sense_target, out1, out2)

    def run(self, routine):
        self._reset()

        while self.moves < self.max_moves:
            routine()

In [18]:
class Prog:

    def _progn(self, *args):
        for arg in args:
            arg()

    def prog2(self, out1, out2):
        return partial(self._progn, out1, out2)

    def prog3(self, out1, out2, out3):
        return partial(self._progn, out1, out2, out3)

In [19]:
max_moves = 750

robot = RobotController(max_moves)
robot.traverse_map(target_map)

print("Start position:", robot.row_start, robot.col_start)
print("Map size:", robot.matrix_row, robot.matrix_col)

Start position: 0 0
Map size: 30 32


In [20]:
pset = gp.PrimitiveSet("MAIN", 0)

pset.addPrimitive(robot.if_target_ahead, 2)
pset.addPrimitive(Prog().prog2, 2)
pset.addPrimitive(Prog().prog3, 3)

pset.addTerminal(robot.move_forward)
pset.addTerminal(robot.turn_left)
pset.addTerminal(robot.turn_right)

In [21]:
if "FitnessMaxRobot" not in creator.__dict__:
    creator.create("FitnessMaxRobot", base.Fitness, weights=(1.0,))

if "IndividualRobot" not in creator.__dict__:
    creator.create("IndividualRobot", gp.PrimitiveTree, fitness=creator.FitnessMaxRobot)

In [22]:
toolbox = base.Toolbox()

toolbox.register("expr", gp.genFull, pset=pset, min_=1, max_=2)
toolbox.register("individual", tools.initIterate, creator.IndividualRobot, toolbox.expr)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("compile", gp.compile, pset=pset)

In [23]:
def eval_func(individual):
    routine = toolbox.compile(expr=individual)
    robot.run(routine)
    return robot.consumed,

In [24]:
toolbox.register("evaluate", eval_func)
toolbox.register("select", tools.selTournament, tournsize=7)
toolbox.register("mate", gp.cxOnePoint)

toolbox.register("expr_mut", gp.genFull, min_=0, max_=2)
toolbox.register("mutate", gp.mutUniform, expr=toolbox.expr_mut, pset=pset)

In [25]:
random.seed(7)

population = toolbox.population(n=400)
hall_of_fame = tools.HallOfFame(1)

In [26]:
stats = tools.Statistics(lambda x: x.fitness.values)

stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)

In [27]:
population, log = algorithms.eaSimple(
    population,
    toolbox,
    cxpb=0.4,
    mutpb=0.3,
    ngen=50,
    stats=stats,
    halloffame=hall_of_fame,
    verbose=True
)

gen	nevals	avg 	std    	min	max
0  	400   	2.78	6.52393	0  	29 
1  	229   	8.5325	10.8944	0  	29 
2  	238   	15.5975	13.3742	0  	44 
3  	233   	19.585 	13.24  	0  	44 
4  	217   	20.485 	13.3819	0  	63 
5  	241   	22.3125	15.2586	0  	63 
6  	234   	23.5675	18.1254	0  	63 
7  	219   	29.38  	21.1466	0  	63 
8  	247   	32.3625	24.3938	0  	63 
9  	228   	40.6325	25.0521	0  	63 
10 	234   	39.6575	25.5122	0  	63 
11 	221   	41.5525	25.4822	0  	63 
12 	233   	41.6   	25.3481	0  	63 
13 	244   	39.69  	26.2793	0  	63 
14 	225   	42.445 	25.2493	0  	63 
15 	234   	41.6575	25.8723	0  	63 
16 	235   	41.4325	26.178 	0  	63 
17 	239   	41.555 	25.9135	0  	63 
18 	232   	45.475 	23.8304	0  	63 
19 	235   	44.0675	25.2922	0  	63 
20 	219   	44.5375	25.6049	0  	63 
21 	244   	43.64  	25.6097	0  	63 
22 	263   	42.505 	25.4267	0  	63 
23 	242   	45.75  	24.7679	0  	63 
24 	217   	45.965 	24.6091	0  	63 
25 	216   	45.135 	25.3516	0  	63 
26 	234   	45.545 	24.7965	0  	63 
27 	219   	46.29  	24.9772	

In [28]:
best_individual = hall_of_fame[0]

print("Best individual:")
print(best_individual)

print("\nTargets consumed:")
print(best_individual.fitness.values[0])

Best individual:
prog3(if_target_ahead(if_target_ahead(turn_right, move_forward), turn_left), prog3(turn_left, if_target_ahead(move_forward, prog3(prog2(prog2(turn_left, move_forward), if_target_ahead(move_forward, turn_right)), if_target_ahead(move_forward, turn_right), move_forward)), move_forward), if_target_ahead(move_forward, turn_right))

Targets consumed:
91.0
